# AuthentiScan — A3 session 1 of 6

Runs: **resnet50_fe, resnet50_ft** | worst-case (30 epochs) ~4.3 h | per-run time budget 330 min

Plan: `sem8_major/implementation_plan.md` §7a A3. Each config runs as its own
subprocess and a failure does not stop the session. Before running, check the
quota page shows enough GPU hours left for the worst case above.

**After it finishes:** download `results_a3_s1.zip` from the Output tab and send
it back — rows are merged into the repo with `code/merge_runs.py`, never retyped.


In [ ]:
# 1. Code: clone the private repo (Kaggle Secret GITHUB_PAT) or pull if already there
import os, subprocess
from kaggle_secrets import UserSecretsClient

PAT = UserSecretsClient().get_secret("GITHUB_PAT")
REPO_DIR = "/kaggle/working/sm7"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone",
                    f"https://{PAT}@github.com/rohityaduvxnshi/sm7.git", REPO_DIR],
                   check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
os.chdir(f"{REPO_DIR}/sem8_major")
print(subprocess.run(["git", "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout)


In [ ]:
# 2. Extras only - never upgrade Kaggle's torch/torchvision (CUDA build is matched)
!pip install -q timm grad-cam

# Accelerator consistency: every matrix run must use the SAME GPU, or Paper 2's
# training-time comparison across architectures compares hardware, not models.
# Calibration and all earlier runs used T4 (Settings -> Accelerator -> GPU T4 x2).
import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU"
print("GPU:", gpu)
if "T4" not in gpu:
    print(f"\n*** WARNING: this session is on {gpu}, not the T4 used for calibration")
    print("*** Stop, switch Accelerator to GPU T4 x2, and restart - otherwise this")
    print("*** run's training time is not comparable with the rest of the matrix.")


In [ ]:
# 3. Pre-flight: matrix configs must be in matrix state, or a subset run could
# silently produce a wrong row in the Paper 2 results table (plan 7a A3).
import yaml, pathlib
CONFIGS = ['resnet50_fe', 'resnet50_ft']
for name in CONFIGS:
    p = pathlib.Path(f"configs/{name}.yaml")
    c = yaml.safe_load(p.read_text(encoding="utf-8"))
    assert c.get("smoke_subset") is None, f"{p} has smoke_subset set"
    assert c["output"]["runs_csv"].endswith("runs.csv"), p
    print(f"OK {name}: {c['model']} {c['mode']} {c['optimizer']} lr={c['lr']} "
          f"bs={c['batch_size']} max_epochs={c['max_epochs']}")


In [ ]:
# 4. The session. One subprocess per config; a failure does not stop the rest.
!python code/run_session.py configs/resnet50_fe.yaml configs/resnet50_ft.yaml \
    --data-root /kaggle/input/cifake-real-and-ai-generated-synthetic-images --results-dir /kaggle/working/results


In [ ]:
# 5. What this session produced
import pandas as pd
df = pd.read_csv("/kaggle/working/results/runs.csv")
cols = ["run_id", "model", "mode", "best_epoch", "stop_reason", "train_time_min",
        "val_acc", "test_acc", "test_auc"]
display(df[cols])
print("\nSanity (gate G3): fine-tuned CNN val_acc should be mid-90s; anything near")
print("0.50 means a pipeline bug - stop and debug rather than spending more quota.")


In [ ]:
# 6. Package for download: results (incl. checkpoints) + nothing else
!cd /kaggle/working && zip -qr results_a3_s1.zip results && ls -lh results_a3_s1.zip
